# Hopper NEAT — Hyperparameter Search Analysis

Sweep: `pop_size` × `generation_limit` × `seed` → 27 runs total.

**Part A** — Results CSV (final best fitness per run)  
**Part B** — MLflow per-generation metrics (training curves, species dynamics, etc.)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

---
# Part A — Results CSV Analysis

In [ ]:
df = pd.read_csv('results.csv')
df.head()

## A1 — Overall Summary

In [ ]:
df[['best_fitness', 'time_s']].describe().round(2)

## A2 — Descriptive Statistics by `pop_size`

In [ ]:
df.groupby('pop_size')['best_fitness'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).round(2)

## A3 — Descriptive Statistics by `generation_limit`

In [ ]:
df.groupby('generation_limit')['best_fitness'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).round(2)

## A4 — Descriptive Statistics by `seed`

In [ ]:
df.groupby('seed')['best_fitness'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).round(2)

## A5 — Descriptive Statistics by (`pop_size`, `generation_limit`)

In [ ]:
df.groupby(['pop_size', 'generation_limit'])['best_fitness'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).round(2)

## A6 — Pivot Tables

In [ ]:
print('Mean Fitness (pop_size × generation_limit):')
display(df.pivot_table(index='pop_size', columns='generation_limit',
                       values='best_fitness', aggfunc=['mean', 'std']).round(2))

print('\nMean Runtime (seconds):')
display(df.pivot_table(index='pop_size', columns='generation_limit',
                       values='time_s', aggfunc='mean').round(1))

## A7 — Best Configuration

In [ ]:
best_row = df.loc[df['best_fitness'].idxmax()]
print('Best single run:')
print(f"  pop_size={int(best_row['pop_size'])}, gen={int(best_row['generation_limit'])}, "
      f"seed={int(best_row['seed'])}, fitness={best_row['best_fitness']:.2f}")

mean_by_config = df.groupby(['pop_size', 'generation_limit'])['best_fitness'].mean()
bc = mean_by_config.idxmax()
print(f'\nBest config by mean fitness: pop_size={bc[0]}, gen={bc[1]}, '
      f'mean={mean_by_config.max():.2f}')

## A8 — Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, col in zip(axes, ['pop_size', 'generation_limit', 'seed']):
    df.boxplot(column='best_fitness', by=col, ax=ax)
    ax.set_title(f'by {col}')
    ax.set_xlabel(col)
axes[0].set_ylabel('best_fitness')
plt.suptitle('Final Best Fitness Distributions')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax = axes[0]
mean_p = df.pivot_table(index='pop_size', columns='generation_limit', values='best_fitness', aggfunc='mean')
std_p  = df.pivot_table(index='pop_size', columns='generation_limit', values='best_fitness', aggfunc='std')
x = np.arange(len(mean_p.index)); w = 0.25
for i, gen in enumerate(mean_p.columns):
    ax.bar(x + i*w, mean_p[gen], w, yerr=std_p[gen], capsize=4, label=f'gen={gen}')
ax.set_xticks(x + w); ax.set_xticklabels(mean_p.index)
ax.set_xlabel('pop_size'); ax.set_ylabel('Mean Fitness (± std)')
ax.set_title('Mean Fitness by Config'); ax.legend()

# Heatmap
ax = axes[1]
sns.heatmap(mean_p, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Mean Fitness Heatmap')
plt.tight_layout(); plt.show()

---
# Part B — MLflow Per-Generation Metrics

Extract training curves from the MLflow SQLite database.

In [ ]:
import sqlite3

MLFLOW_DB = '/home/20240503/tensorneat/mlflow.db'
con = sqlite3.connect(MLFLOW_DB)

# Get runs from the hyperparam search only
runs = pd.read_sql_query("""
    SELECT run_uuid, name
    FROM runs
    WHERE name LIKE 'hopper_neat_pop%_seed%'
""", con)

# Get params for each run
params = pd.read_sql_query("""
    SELECT run_uuid, key, value FROM params
    WHERE run_uuid IN (SELECT run_uuid FROM runs WHERE name LIKE 'hopper_neat_pop%_seed%')
""", con)
params_wide = params.pivot(index='run_uuid', columns='key', values='value').reset_index()

# Get all per-generation metrics
metrics = pd.read_sql_query("""
    SELECT m.run_uuid, m.key, m.value, m.step
    FROM metrics m
    WHERE m.run_uuid IN (SELECT run_uuid FROM runs WHERE name LIKE 'hopper_neat_pop%_seed%')
    ORDER BY m.run_uuid, m.key, m.step
""", con)

con.close()

# Merge run info
metrics = metrics.merge(runs, on='run_uuid')
metrics = metrics.merge(params_wide[['run_uuid', 'pop_size', 'generation_limit', 'seed']], on='run_uuid')
for col in ['pop_size', 'generation_limit', 'seed']:
    metrics[col] = metrics[col].astype(int)

print(f'Loaded {len(metrics):,} metric rows across {runs.shape[0]} runs')
print(f'Metrics available: {sorted(metrics["key"].unique())}')

## B1 — Available Metrics Overview

In [ ]:
metrics.groupby('key').agg(
    n_rows=('value', 'count'),
    mean=('value', 'mean'),
    std=('value', 'std'),
    min=('value', 'min'),
    max=('value', 'max'),
).round(2)

## B2 — Fitness Training Curves (max fitness per generation)

Each line = one run, colored by `pop_size`, paneled by `generation_limit`.

In [ ]:
fit_max = metrics[metrics['key'] == 'fitness/max'].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
colors = {512: '#1f77b4', 1024: '#ff7f0e', 2048: '#2ca02c'}

for ax, gen in zip(axes, [100, 200, 300]):
    subset = fit_max[fit_max['generation_limit'] == gen]
    for _, grp in subset.groupby(['pop_size', 'seed']):
        ps = grp['pop_size'].iloc[0]
        ax.plot(grp['step'], grp['value'], alpha=0.6, color=colors[ps], linewidth=1)
    ax.set_title(f'generation_limit = {gen}')
    ax.set_xlabel('Generation')

axes[0].set_ylabel('Max Fitness')
from matplotlib.lines import Line2D
legend_elems = [Line2D([0],[0], color=c, lw=2, label=f'pop={p}') for p, c in colors.items()]
axes[-1].legend(handles=legend_elems, loc='lower right')
plt.suptitle('Max Fitness Training Curves', fontsize=14)
plt.tight_layout(); plt.show()

## B3 — Mean Fitness Training Curves (with seed-averaged ribbon)

In [ ]:
fit_mean = metrics[metrics['key'] == 'fitness/mean'].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, gen in zip(axes, [100, 200, 300]):
    subset = fit_mean[fit_mean['generation_limit'] == gen]
    for ps, color in colors.items():
        grp = subset[subset['pop_size'] == ps]
        agg = grp.groupby('step')['value'].agg(['mean', 'std']).reset_index()
        ax.plot(agg['step'], agg['mean'], color=color, label=f'pop={ps}')
        ax.fill_between(agg['step'], agg['mean'] - agg['std'],
                        agg['mean'] + agg['std'], alpha=0.15, color=color)
    ax.set_title(f'gen_limit = {gen}')
    ax.set_xlabel('Generation')

axes[0].set_ylabel('Mean Pop. Fitness (± 1 std across seeds)')
axes[-1].legend(loc='lower right')
plt.suptitle('Mean Fitness Curves (seed-averaged)', fontsize=14)
plt.tight_layout(); plt.show()

## B4 — Species Count Over Generations

In [ ]:
species = metrics[metrics['key'] == 'neat/n_species'].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, gen in zip(axes, [100, 200, 300]):
    subset = species[species['generation_limit'] == gen]
    for ps, color in colors.items():
        grp = subset[subset['pop_size'] == ps]
        agg = grp.groupby('step')['value'].agg(['mean', 'std']).reset_index()
        ax.plot(agg['step'], agg['mean'], color=color, label=f'pop={ps}')
        ax.fill_between(agg['step'], agg['mean'] - agg['std'],
                        agg['mean'] + agg['std'], alpha=0.15, color=color)
    ax.set_title(f'gen_limit = {gen}')
    ax.set_xlabel('Generation')

axes[0].set_ylabel('Species Count (± 1 std)')
axes[-1].legend(loc='upper right')
plt.suptitle('Species Dynamics', fontsize=14)
plt.tight_layout(); plt.show()

## B5 — Genome Complexity (avg connections over generations)

In [ ]:
conns = metrics[metrics['key'] == 'neat/avg_genome_n_conns'].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, gen in zip(axes, [100, 200, 300]):
    subset = conns[conns['generation_limit'] == gen]
    for ps, color in colors.items():
        grp = subset[subset['pop_size'] == ps]
        agg = grp.groupby('step')['value'].agg(['mean', 'std']).reset_index()
        ax.plot(agg['step'], agg['mean'], color=color, label=f'pop={ps}')
        ax.fill_between(agg['step'], agg['mean'] - agg['std'],
                        agg['mean'] + agg['std'], alpha=0.15, color=color)
    ax.set_title(f'gen_limit = {gen}')
    ax.set_xlabel('Generation')

axes[0].set_ylabel('Avg Connections per Genome (± 1 std)')
axes[-1].legend(loc='lower right')
plt.suptitle('Genome Complexity Growth', fontsize=14)
plt.tight_layout(); plt.show()

## B6 — Cost Time per Generation

In [ ]:
cost = metrics[metrics['key'] == 'cost_time_ms'].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, gen in zip(axes, [100, 200, 300]):
    subset = cost[cost['generation_limit'] == gen]
    for ps, color in colors.items():
        grp = subset[subset['pop_size'] == ps]
        agg = grp.groupby('step')['value'].agg(['mean', 'std']).reset_index()
        ax.plot(agg['step'], agg['mean'], color=color, label=f'pop={ps}')
    ax.set_title(f'gen_limit = {gen}')
    ax.set_xlabel('Generation')

axes[0].set_ylabel('Cost Time (ms)')
axes[-1].legend(loc='upper right')
plt.suptitle('Per-Generation Compute Cost', fontsize=14)
plt.tight_layout(); plt.show()

## B7 — Fitness Variance Over Training

How the within-population fitness standard deviation evolves.

In [ ]:
fit_std = metrics[metrics['key'] == 'fitness/std'].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, gen in zip(axes, [100, 200, 300]):
    subset = fit_std[fit_std['generation_limit'] == gen]
    for ps, color in colors.items():
        grp = subset[subset['pop_size'] == ps]
        agg = grp.groupby('step')['value'].agg(['mean', 'std']).reset_index()
        ax.plot(agg['step'], agg['mean'], color=color, label=f'pop={ps}')
        ax.fill_between(agg['step'], agg['mean'] - agg['std'],
                        agg['mean'] + agg['std'], alpha=0.15, color=color)
    ax.set_title(f'gen_limit = {gen}')
    ax.set_xlabel('Generation')

axes[0].set_ylabel('Population Fitness Std Dev')
axes[-1].legend(loc='upper right')
plt.suptitle('Within-Population Fitness Diversity', fontsize=14)
plt.tight_layout(); plt.show()

## B8 — Stagnation Counter

In [ ]:
stag = metrics[metrics['key'] == 'stability/stagnation_counter'].copy()

if not stag.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    for ax, gen in zip(axes, [100, 200, 300]):
        subset = stag[stag['generation_limit'] == gen]
        for ps, color in colors.items():
            grp = subset[subset['pop_size'] == ps]
            agg = grp.groupby('step')['value'].agg(['mean', 'std']).reset_index()
            ax.plot(agg['step'], agg['mean'], color=color, label=f'pop={ps}')
        ax.set_title(f'gen_limit = {gen}')
        ax.set_xlabel('Generation')
    axes[0].set_ylabel('Stagnation Counter')
    axes[-1].legend()
    plt.suptitle('Stagnation Counter Over Generations', fontsize=14)
    plt.tight_layout(); plt.show()
else:
    print('No stagnation counter data found.')

## B9 — Summary Statistics: MLflow Metrics at Final Generation

In [ ]:
# For each run, take the metric value at the last step
last_step = metrics.groupby(['run_uuid', 'key'])['step'].max().reset_index()
last_step.columns = ['run_uuid', 'key', 'last_step']

final_metrics = metrics.merge(last_step, on=['run_uuid', 'key'])
final_metrics = final_metrics[final_metrics['step'] == final_metrics['last_step']]

final_wide = final_metrics.pivot_table(
    index=['pop_size', 'generation_limit', 'seed'],
    columns='key', values='value'
).reset_index()

print('Final-generation metric summary by config:')
display(
    final_wide.groupby(['pop_size', 'generation_limit'])
    .mean(numeric_only=True)
    .round(2)
)